# Day-ahead prediction results

Run this notebook after a training experiment finishes. It loads the latest completed MLflow run, rebuilds that run’s held-out test rows from its recorded Delta versions, and scores them with the logged model and seasonal baseline. No training or manual auction-series selection is needed.

In [ ]:
import json
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.lightgbm
from mlflow.tracking import MlflowClient

# VS Code uses the package installed by uv; a Databricks workspace notebook uses its synced source tree.
try:
    from eu_electricity_forecast.training import _features, _seasonal_baseline, _validate_and_prepare, chronological_split, load_training_frame
except ModuleNotFoundError as exc:
    if exc.name != "eu_electricity_forecast":
        raise
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = notebook_path.split("/src/notebooks/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "src"))
    from eu_electricity_forecast.training import _features, _seasonal_baseline, _validate_and_prepare, chronological_split, load_training_frame

# Use the current Databricks workspace for MLflow tracking from either the UI or VS Code.
mlflow.set_tracking_uri("databricks")
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_PATH = f"/Users/{current_user}/eu-electricity-forecast-day-ahead-price"
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_PATH)
if experiment is None:
    raise ValueError(f"MLflow experiment not found: {EXPERIMENT_PATH}")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)
if not runs:
    raise ValueError(f"No completed training runs found in {EXPERIMENT_PATH}")
selected_run = runs[0]
summary_path = client.download_artifacts(selected_run.info.run_id, "experiment_summary.json")
with open(summary_path) as summary_file:
    run_summary = json.load(summary_file)

# Recreate the exact chronological split from the source Delta versions logged by this run.
training_frame = load_training_frame(
    spark,
    run_summary["source_table"],
    run_summary["series_by_zone"],
    int(selected_run.data.params["resolution_minutes"]),
    run_summary["source_delta_versions"],
)
training_frame = _validate_and_prepare(
    training_frame, run_summary["series_by_zone"], int(selected_run.data.params["resolution_minutes"])
)
partitions, _ = chronological_split(training_frame)
train_frame, test_frame = partitions["train"], partitions["test"]
zone_categories = sorted(train_frame["bidding_zone"].dropna().unique().tolist())
test_features, _ = _features(test_frame, zone_categories)
logged_model = mlflow.lightgbm.load_model(f"runs:/{selected_run.info.run_id}/model")
test_predictions = test_frame[
    ["bidding_zone", "target_interval_start_utc", "label_price_eur_mwh", "target_auction_series_key"]
].copy()
test_predictions["model_prediction"] = logged_model.predict(test_features)
test_predictions["seasonal_baseline_prediction"] = _seasonal_baseline(train_frame, test_frame)
test_predictions["target_interval_start_utc"] = pd.to_datetime(test_predictions["target_interval_start_utc"], utc=True)
print(f"MLflow run: {selected_run.info.run_id}")
print(f"Resolution: {selected_run.data.params.get('resolution_minutes')} minutes")
print(f"Test rows: {len(test_predictions):,}; zones: {sorted(test_predictions['bidding_zone'].unique())}")
display(pd.DataFrame([
    {"model": "LightGBM", "test MAE (€/MWh)": selected_run.data.metrics.get("test_model_mae")},
    {"model": "Seasonal baseline", "test MAE (€/MWh)": selected_run.data.metrics.get("test_seasonal_baseline_mae")},
]))

## Actual prices and predictions by zone

Each panel compares held-out actual prices with the trained LightGBM model and seasonal baseline. Change `PLOT_DAYS` to view more or less of the test period. Each zone has its own price scale.

In [ ]:
PLOT_DAYS = 14
latest_test_time = test_predictions["target_interval_start_utc"].max()
plot_start = latest_test_time - pd.Timedelta(days=PLOT_DAYS)
zones = sorted(test_predictions["bidding_zone"].dropna().unique())
fig, axes = plt.subplots(len(zones), 1, figsize=(16, 3.8 * len(zones)), sharex=True, squeeze=False)

for zone, ax in zip(zones, axes[:, 0]):
    chart = (
        test_predictions[
            (test_predictions["bidding_zone"] == zone)
            & (test_predictions["target_interval_start_utc"] >= plot_start)
        ]
        .sort_values("target_interval_start_utc")
        .set_index("target_interval_start_utc")
    )
    chart[["label_price_eur_mwh", "model_prediction", "seasonal_baseline_prediction"]].plot(
        ax=ax, linewidth=1.1,
        title=(
            f"{zone} | MAE: model "
            f"{(chart['model_prediction'] - chart['label_price_eur_mwh']).abs().mean():.1f}, "
            f"baseline {(chart['seasonal_baseline_prediction'] - chart['label_price_eur_mwh']).abs().mean():.1f} €/MWh"
        ),
    )
    ax.set_ylabel("€/MWh")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", ncol=3)

axes[-1, 0].set_xlabel("Delivery interval start (UTC)")
fig.suptitle(f"Held-out day-ahead price predictions | latest {PLOT_DAYS} test days", y=1.01)
fig.tight_layout()
display(fig)
plt.close(fig)